# 06 - LangChain 向量存储集成（Vector Stores）

## 学习目标
- 掌握 Chroma 向量数据库的集成与持久化
- 使用 Qdrant 进行云端向量存储和集合管理
- 使用 Pinecone 无服务器索引和元数据过滤
- 理解 as_retriever() 的三种搜索类型
- 掌握 MultiVectorRetriever 父文档检索模式
- 对比不同搜索参数对结果的影响

In [ ]:
# 安装必要依赖（如未安装请取消注释）
# !pip install langchain langchain-community langchain-openai langchain-chroma chromadb qdrant-client pinecone-client langchain-pinecone langchain-qdrant

## 1. Chroma 集成

Chroma 是一个开源的嵌入式向量数据库，支持本地持久化存储。

In [ ]:
import os
import tempfile

# 准备测试数据
test_texts = [
    "LangChain 是一个用于开发 LLM 应用的框架",
    "向量数据库用于存储和检索文档嵌入向量",
    "Python 是一门流行的编程语言",
    "RAG 系统结合检索和生成两个步骤",
    "Chroma 是一个开源的嵌入式向量数据库",
    "机器学习模型需要大量训练数据",
    "检索增强生成提升了 LLM 回答的准确性",
    "Docker 用于容器化部署应用",
]

test_metadatas = [
    {"category": "AI", "topic": "langchain"},
    {"category": "AI", "topic": "vectordb"},
    {"category": "Programming", "topic": "python"},
    {"category": "AI", "topic": "rag"},
    {"category": "AI", "topic": "chroma"},
    {"category": "AI", "topic": "ml"},
    {"category": "AI", "topic": "rag"},
    {"category": "DevOps", "topic": "docker"},
]

print(f"准备了 {len(test_texts)} 条测试文档")
print(f"分类分布:")
from collections import Counter
cats = Counter(m["category"] for m in test_metadatas)
for cat, count in cats.items():
    print(f"  {cat}: {count} 条")

In [ ]:
# === Chroma 集成: 使用 FakeEmbeddings 避免 API 调用 ===
# 在生产环境中使用 OpenAIEmbeddings 或其他真实嵌入模型

try:
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings
    
    CHROMA_AVAILABLE = True
    print("langchain-chroma 已安装")
except ImportError:
    CHROMA_AVAILABLE = False
    print("langchain-chroma 未安装，使用模拟演示")
    print("请运行: pip install langchain-chroma chromadb")

if CHROMA_AVAILABLE:
    from langchain_chroma import Chroma
    from langchain_core.documents import Document
    
    # 使用 FakeEmbeddings 进行本地演示（无需 API key）
    from langchain_core.embeddings import FakeEmbeddings
    embeddings = FakeEmbeddings(size=384)
    
    docs = [
        Document(page_content=text, metadata=meta)
        for text, meta in zip(test_texts, test_metadatas)
    ]
    
    # === 创建 Chroma 向量存储（持久化模式） ===
    persist_dir = tempfile.mkdtemp(prefix="chroma_persist_")
    print(f"持久化目录: {persist_dir}")
    
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_dir,
        collection_name="rag_tutorial",
        collection_metadata={"hnsw:space": "cosine"},  # 距离度量
    )
    
    print(f"\n向量存储创建成功")
    print(f"集合名称: {vectorstore._collection.name}")
    print(f"文档数量: {vectorstore._collection.count()}")
    
    # === 相似性搜索 ===
    print("\n" + "=" * 60)
    print("相似性搜索: 'LLM 应用开发'")
    print("=" * 60)
    
    results = vectorstore.similarity_search(
        query="LLM 应用开发",
        k=3,
    )
    
    for i, doc in enumerate(results):
        print(f"\n[{i+1}] {doc.page_content}")
        print(f"    元数据: {doc.metadata}")
    
    # === 带分数的相似性搜索 ===
    print("\n" + "=" * 60)
    print("带分数的相似性搜索: '向量检索'")
    print("=" * 60)
    
    results_with_scores = vectorstore.similarity_search_with_score(
        query="向量检索",
        k=4,
    )
    
    for i, (doc, score) in enumerate(results_with_scores):
        print(f"\n[{i+1}] 分数: {score:.4f} | {doc.page_content}")
        print(f"    元数据: {doc.metadata}")
    
    # === 元数据过滤搜索 ===
    print("\n" + "=" * 60)
    print("元数据过滤搜索: category='AI'")
    print("=" * 60)
    
    filtered_results = vectorstore.similarity_search(
        query="编程",
        k=3,
        filter={"category": "AI"},
    )
    
    for i, doc in enumerate(filtered_results):
        print(f"[{i+1}] [{doc.metadata['category']}] {doc.page_content}")
    
    # === 持久化验证 ===
    print("\n" + "=" * 60)
    print("持久化验证: 重新加载向量存储")
    print("=" * 60)
    
    # 从磁盘重新加载
    vectorstore2 = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name="rag_tutorial",
    )
    
    reload_results = vectorstore2.similarity_search("LLM", k=2)
    print(f"重新加载后搜索 'LLM' 得到 {len(reload_results)} 个结果")
    for doc in reload_results:
        print(f"  - {doc.page_content[:50]}...")
    
    # 清理
    import shutil
    shutil.rmtree(persist_dir, ignore_errors=True)
    print(f"\n已清理持久化目录")

else:
    print("\n# Chroma 概念演示代码:")
    print("from langchain_chroma import Chroma")
    print("from langchain_openai import OpenAIEmbeddings")
    print()
    print("vectorstore = Chroma.from_documents(")
    print("    documents=docs,")
    print("    embedding=OpenAIEmbeddings(),")
    print("    persist_directory='./chroma_db',")
    print(")")
    print()
    print("results = vectorstore.similarity_search('query', k=3)")
    print("results_with_score = vectorstore.similarity_search_with_score('query', k=3)")

## 2. Qdrant 集成

Qdrant 是一个高性能向量搜索引擎，支持本地部署和云端托管。

In [ ]:
print("=== Qdrant 集成 ===\n")

print("# Qdrant 支持两种部署模式:")
print()
print("# 模式1: 本地模式（内存/磁盘）")
print("from langchain_qdrant import QdrantVectorStore")
print("from qdrant_client import QdrantClient")
print()
print("client = QdrantClient(':memory:')  # 内存模式")
print("# client = QdrantClient(path='./qdrant_db')  # 磁盘持久化")
print()
print("vectorstore = QdrantVectorStore.from_documents(")
print("    documents=docs,")
print("    embedding=embeddings,")
print("    client=client,")
print("    collection_name='my_collection',")
print(")")

print("\n" + "=" * 60)

print("# 模式2: 云端连接")
print("from langchain_qdrant import QdrantVectorStore")
print("from qdrant_client import QdrantClient")
print()
print("client = QdrantClient(")
print("    url='https://your-cluster.qdrant.cloud',")
print("    api_key='your-api-key',")
print(")")
print()
print("vectorstore = QdrantVectorStore(")
print("    client=client,")
print("    collection_name='production_docs',")
print("    embedding=embeddings,")
print(")")

print("\n" + "=" * 60)

print("# 集合管理操作:")
print("# 创建集合")
print("from qdrant_client.http.models import VectorParams, Distance")
print()
print("client.create_collection(")
print("    collection_name='my_collection',")
print("    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),")
print(")")
print()
print("# 列出所有集合")
print("collections = client.get_collections()")
print()
print("# 删除集合")
print("client.delete_collection('my_collection')")

print("\n" + "=" * 60)

print("# Qdrant 的相似性搜索:")
print("results = vectorstore.similarity_search(")
print("    query='你的查询',")
print("    k=5,")
print("    filter={")
print("        'must': [")
print("            {'key': 'category', 'match': {'value': 'AI'}},")
print("        ]")
print("    },")
print(")")

## 3. Pinecone 集成

Pinecone 是一个托管的、无服务器向量数据库，支持元数据过滤和命名空间。

In [ ]:
print("=== Pinecone 集成 ===\n")

print("# Pinecone 无服务器索引设置:")
print("from pinecone import Pinecone, ServerlessSpec")
print()
print("pc = Pinecone(api_key='your-api-key')")
print()
print("# 创建无服务器索引")
print("pc.create_index(")
print("    name='langchain-rag',")
print("    dimension=1536,           # OpenAI text-embedding-ada-002 维度")
print("    metric='cosine',")
print("    spec=ServerlessSpec(")
print("        cloud='aws',")
print("        region='us-east-1'")
print("    )")
print(")")

print("\n" + "=" * 60)

print("# 连接到 Pinecone 索引并创建向量存储:")
print("from langchain_pinecone import PineconeVectorStore")
print("from langchain_openai import OpenAIEmbeddings")
print()
print("embeddings = OpenAIEmbeddings()")
print("index = pc.Index('langchain-rag')")
print()
print("vectorstore = PineconeVectorStore(")
print("    index=index,")
print("    embedding=embeddings,")
print("    namespace='chinese_docs',  # 命名空间隔离")
print(")  # text_key 默认为 'text'")

print("\n" + "=" * 60)

print("# Pinecone 的元数据过滤搜索:")
print("results = vectorstore.similarity_search(")
print("    query='LangChain 教程',")
print("    k=5,")
print("    filter={")
print("        '$and': [")
print("            {'category': {'$eq': 'AI'}},")
print("            {'language': {'$in': ['zh', 'en']}},")
print("            {'timestamp': {'$gte': '2024-01-01'}},")
print("        ]")
print("    },")
print("    namespace='chinese_docs',")
print(")")

print("\n" + "=" * 60)

print("# 命名空间管理:")
print("# 列出所有命名空间")
print("stats = index.describe_index_stats()")
print("print(stats['namespaces'])")
print()
print("# 删除命名空间中的所有向量")
print("index.delete(delete_all=True, namespace='old_namespace')")

## 4. as_retriever() - 三种搜索类型详解

as_retriever() 将向量存储包装为检索器，支持三种搜索类型，每种返回不同的结果。

In [ ]:
try:
    from langchain_chroma import Chroma
    from langchain_core.documents import Document
    from langchain_core.embeddings import FakeEmbeddings
    
    embeddings = FakeEmbeddings(size=384)
    
    # 准备更丰富的测试数据来展示不同搜索类型的差异
    diverse_texts = [
        "LangChain 框架用于构建 LLM 应用，支持链式调用",
        "LangChain 提供了文档加载器、向量存储和检索器",
        "LangChain 的 LCEL 语法使得链的组合更加方便",
        "Python 编程语言基础教程，包括变量和函数",
        "Python 高级特性：装饰器、生成器和上下文管理器",
        "向量数据库 Chroma 的安装和配置指南",
        "向量数据库 Qdrant 的高性能检索能力",
        "RAG 系统设计模式与最佳实践",
    ]
    
    diverse_metadatas = [
        {"category": "AI", "level": "beginner"},
        {"category": "AI", "level": "intermediate"},
        {"category": "AI", "level": "advanced"},
        {"category": "Programming", "level": "beginner"},
        {"category": "Programming", "level": "advanced"},
        {"category": "AI", "level": "beginner"},
        {"category": "AI", "level": "advanced"},
        {"category": "AI", "level": "intermediate"},
    ]
    
    docs = [
        Document(page_content=text, metadata=meta)
        for text, meta in zip(diverse_texts, diverse_metadatas)
    ]
    
    persist_dir = tempfile.mkdtemp(prefix="chroma_search_types_")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    
    query = "LangChain LLM 应用开发"
    
    # === 类型1: similarity - 默认余弦相似度 ===
    print("=" * 70)
    print("搜索类型1: similarity（默认）")
    print("策略: 返回与查询最相似的 k 个文档，按余弦相似度排序")
    print("=" * 70)
    
    retriever_sim = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4},
    )
    
    sim_results = retriever_sim.invoke(query)
    print(f"\n查询: '{query}'")
    print(f"返回 {len(sim_results)} 个结果:\n")
    for i, doc in enumerate(sim_results):
        print(f"  [{i+1}] {doc.page_content}")
        print(f"      元数据: {doc.metadata}")
    
    # === 类型2: mmr - 最大边际相关性 ===
    print("\n" + "=" * 70)
    print("搜索类型2: mmr（Maximum Marginal Relevance）")
    print("策略: 平衡相关性和多样性，lambda_mult 控制多样性权重")
    print("=" * 70)
    
    retriever_mmr_low_div = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 4,
            "fetch_k": 8,         # 先获取 8 个候选
            "lambda_mult": 0.2,   # 低 lambda = 更注重多样性
        },
    )
    
    mmr_low_results = retriever_mmr_low_div.invoke(query)
    print(f"\nMMR (lambda_mult=0.2, 强调多样性):")
    for i, doc in enumerate(mmr_low_results):
        print(f"  [{i+1}] {doc.page_content}")
    
    # === mmr 高相关性版本 ===
    retriever_mmr_high_rel = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 4,
            "fetch_k": 8,
            "lambda_mult": 0.8,   # 高 lambda = 更注重相关性
        },
    )
    
    mmr_high_results = retriever_mmr_high_rel.invoke(query)
    print(f"\nMMR (lambda_mult=0.8, 强调相关性):")
    for i, doc in enumerate(mmr_high_results):
        print(f"  [{i+1}] {doc.page_content}")
    
    # === 类型3: similarity_score_threshold ===
    print("\n" + "=" * 70)
    print("搜索类型3: similarity_score_threshold")
    print("策略: 只返回相似度分数超过阈值的文档")
    print("=" * 70)
    
    retriever_threshold = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 5,
            "score_threshold": 0.1,
        },
    )
    
    threshold_results = retriever_threshold.invoke("Docker 容器")
    print(f"\n查询: 'Docker 容器'（与向量存储内容不相关）")
    print(f"阈值: 0.1")
    print(f"返回 {len(threshold_results)} 个结果:")
    if len(threshold_results) == 0:
        print("  (无结果 - 所有文档分数均低于阈值)")
    else:
        for i, doc in enumerate(threshold_results):
            print(f"  [{i+1}] {doc.page_content}")
    
    threshold_results2 = retriever_threshold.invoke("LangChain LCEL")
    print(f"\n查询: 'LangChain LCEL'（与向量存储内容相关）")
    print(f"返回 {len(threshold_results2)} 个结果:")
    for i, doc in enumerate(threshold_results2):
        print(f"  [{i+1}] {doc.page_content}")
    
    # 清理
    import shutil
    shutil.rmtree(persist_dir, ignore_errors=True)
    
except Exception as e:
    print(f"注意: 向量存储演示需要 langchain-chroma")
    print(f"错误: {e}")
    print()
    print("# 替代方案 - 概念演示代码:")
    print("retriever = vectorstore.as_retriever(")
    print("    search_type='mmr',")
    print("    search_kwargs={'k': 4, 'fetch_k': 10, 'lambda_mult': 0.5}")
    print(")")

## 5. search_kwargs 参数详解

所有搜索类型共用的 search_kwargs 参数总结。

In [ ]:
print("=== search_kwargs 参数参考 ===\n")

params_table = [
    ("k", "int", "返回的文档数量", "similarity, mmr, similarity_score_threshold", "4"),
    ("fetch_k", "int", "从向量存储获取的候选文档数（mmr从中选择）", "mmr", "20"),
    ("lambda_mult", "float", "多样性控制因子: 0=最大多样性, 1=最大相关性", "mmr", "0.5"),
    ("score_threshold", "float", "最低相似度分数阈值", "similarity_score_threshold", "0.8"),
    ("filter", "dict", "元数据过滤条件", "全部", "{'category': 'AI'}"),
]

print(f"{'参数':<20} {'类型':<8} {'说明':<40} {'适用类型':<30} {'默认值':<10}")
print("-" * 110)
for name, ptype, desc, applies, default in params_table:
    print(f"{name:<20} {ptype:<8} {desc:<40} {applies:<30} {default:<10}")

print("\n# 完整示例 - 组合使用多个参数:")
print()
print("retriever = vectorstore.as_retriever(")
print("    search_type='similarity_score_threshold',")
print("    search_kwargs={")
print("        'k': 5,")
print("        'score_threshold': 0.75,")
print("        'filter': {")
print("            '$and': [")
print("                {'category': {'$eq': 'AI'}},")
print("                {'language': {'$eq': 'zh'}},")
print("                {'year': {'$gte': 2024}},")
print("            ]")
print("        }")
print("    }")
print(")")
print()
print("docs = retriever.invoke('LangChain 教程')")

print("\n# 过滤器支持的运算符:")
print("operators = {")
print("    '$eq': '等于',")
print("    '$ne': '不等于',")
print("    '$gt': '大于',")
print("    '$gte': '大于等于',")
print("    '$lt': '小于',")
print("    '$lte': '小于等于',")
print("    '$in': '在列表中',")
print("    '$nin': '不在列表中',")
print("    '$and': '逻辑与',")
print("    '$or': '逻辑或',")
print("}")
for op, desc in operators.items():
    print(f"  {op}: {desc}")

## 6. MultiVectorRetriever - 父文档检索器模式

MultiVectorRetriever 允许为每个文档存储多个向量，适用于父文档检索模式：将大文档拆分，检索时返回完整父文档。

In [ ]:
print("=== MultiVectorRetriever / 父文档检索器 ===\n")

print("# 父文档检索器的设计思想:")
print("# 1. 将大文档拆分为小段，每一段嵌入为向量")
print("# 2. 检索时用小段匹配查询（提高检索精度）")
print("# 3. 返回时返回完整的父文档（保留上下文）")
print()

print("# 实现方式1: MultiVectorRetriever + 文档存储")
print("from langchain.retrievers import MultiVectorRetriever")
print("from langchain.storage import InMemoryStore")
print("from langchain_chroma import Chroma")
print()

print("# 创建文档存储（保存完整父文档）")
print("store = InMemoryStore()")
print()

print("# 创建向量存储（保存子文档嵌入）")
print("vectorstore = Chroma(embedding_function=embeddings)")
print()

print("# 创建多向量检索器")
print("retriever = MultiVectorRetriever(")
print("    vectorstore=vectorstore,")
print("    docstore=store,")
print("    id_key='doc_id',        # 父文档的唯一标识键")
print("    search_kwargs={'k': 5}, # 检索时的参数")
print(")")

print("\n" + "=" * 60)

print("# 实现方式2: ParentDocumentRetriever（高层封装）")
print("from langchain.retrievers import ParentDocumentRetriever")
print("from langchain_text_splitters import RecursiveCharacterTextSplitter")
print()

print("# 父文档分割器: 大块的上下文")
print("parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)")
print()

print("# 子文档分割器: 小块用于检索")
print("child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)")
print()

print("retriever = ParentDocumentRetriever(")
print("    vectorstore=Chroma(embedding=embeddings),")
print("    docstore=InMemoryStore(),")
print("    child_splitter=child_splitter,")
print("    parent_splitter=parent_splitter,")
print("    search_kwargs={'k': 4},")
print(")")

print("\n" + "=" * 60)

print("# 工作流程:")
print("# retriever.add_documents(full_documents)")
print("#   -> 父文档按 parent_splitter 拆分（大块）")
print("#   -> 每个大块再按 child_splitter 拆分为小块")
print("#   -> 小块嵌入存入向量存储")
print("#   -> 大块存入文档存储（docstore）")
print()
print("# retriever.invoke('查询')")
print("#   -> 用查询在小块向量中检索")
print("#   -> 找到匹配的小块")
print("#   -> 通过 doc_id 查找对应的父文档（大块）")
print("#   -> 返回完整的大块文档")

print("\n" + "=" * 60)

print("# 完整代码示例:")
print("""
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 准备文档
full_docs = [
    Document(
        page_content="这是一篇关于LangChain的完整文章...(长文本)...",
        metadata={"title": "LangChain入门"}
    ),
]

# 创建父文档检索器
retriever = ParentDocumentRetriever(
    vectorstore=Chroma(
        collection_name="parent_docs",
        embedding_function=OpenAIEmbeddings()
    ),
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=400),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=2000),
)

retriever.add_documents(full_docs)

# 检索: 返回的是大块父文档，而非小块
results = retriever.invoke("LangChain的核心概念")
for doc in results:
    print(f"文档长度: {len(doc.page_content)}")  # 接近2000字符
    print(f"标题: {doc.metadata['title']}")
""")

## 7. 三种搜索类型对比总结

用相同查询对比三种搜索类型返回的结果。

In [ ]:
print("=== 搜索类型对比总结 ===\n")

print("情景: 查询 'Python 编程'，向量库包含:")
print("  1. 'Python 入门教程'")
print("  2. 'Python 高级编程'")
print("  3. 'Python 数据分析'")
print("  4. 'LangChain Python 开发'")
print("  5. 'Java 入门教程'")
print()

print("┌─────────────────────┬──────────────────────────────────────────────┐")
print("│ 搜索类型            │ 预期返回结果                                 │")
print("├─────────────────────┼──────────────────────────────────────────────┤")
print("│ similarity (k=3)    │ [1, 2, 3] - 最相关的3个Python文档           │")
print("│ mmr (lambda=0.5,k=3)│ [1, 4, 5] - 多样化的结果，包含相关但不重复   │")
print("│ threshold (0.5,k=3) │ [1, 2, 3] 或 [] - 只返回分数高于0.5的      │")
print("└─────────────────────┴──────────────────────────────────────────────┘")

print("\n# 选择指南:")
print("# - 需要最相关的文档: 使用 similarity")
print("# - 需要多样的信息源: 使用 mmr (lambda_mult < 0.5)")
print("# - 需要质量保证: 使用 similarity_score_threshold")
print("# - 处理元数据过滤: 添加 filter 参数")
print("# - 需要大上下文: 使用 ParentDocumentRetriever")

print("\n# 向量数据库选择指南:")
print("# Chroma:   本地开发、原型验证、小规模数据")
print("# Qdrant:   高性能、支持量化、本地+云端")
print("# Pinecone: 全托管、无服务器、生产环境")
print("# Weaviate: 混合搜索（向量+关键词）、GraphQL API")
print("# Milvus:   超大规模、分布式、GPU 加速")